# Лабораторная работа 3

## Вариант 11

In [1]:
from pathlib import Path
import sys

project_src = Path.cwd() / "src"
if not project_src.exists():
    project_src = Path.cwd().parent / "src"
if project_src.exists():
    sys.path.insert(0, str(project_src))

import numpy as np

from nm.approximation import (
    divided_differences,
    evaluate_cubic_spline,
    lagrange_polynomial,
    natural_cubic_spline_coefficients,
    newton_polynomial,
)

# 3.1. Интерполяционные многочлены Лагранжа и Ньютона

Дано: $y = arcctg(x)$, $X^* = -0.5$.

Узлы интерполяции:

- a) $X_i = -3, -1, 1, 3$;
- б) $X_i = -3, 0, 1, 3$.

Здесь используется ветвь $arcctg(x) = \frac{\pi}{2} - arctg(x)$.

In [2]:
def arcctg(x):
    return np.pi / 2 - np.arctan(x)


x_star = -0.5
interpolation_nodes = {
    "a": np.array([-3.0, -1.0, 1.0, 3.0]),
    "б": np.array([-3.0, 0.0, 1.0, 3.0]),
}

for label, x_nodes in interpolation_nodes.items():
    y_nodes = arcctg(x_nodes)
    lagrange = lagrange_polynomial(x_nodes, y_nodes)
    newton = newton_polynomial(x_nodes, y_nodes)
    differences = divided_differences(x_nodes, y_nodes)

    interpolated = lagrange(x_star)
    real_value = arcctg(x_star)
    error = abs(real_value - interpolated)

    print(f"Вариант {label}")
    print("Xi:", x_nodes)
    print("Yi:", y_nodes)
    print("Разделенные разности:", differences)
    print("Многочлен Лагранжа:")
    print(lagrange)
    print("Многочлен Ньютона, приведенный к стандартному виду:")
    print(newton)
    print(f"P(X*) = {interpolated:.10f}")
    print(f"f(X*) = {real_value:.10f}")
    print(f"|f(X*) - P(X*)| = {error:.10f}")
    print()

    assert np.allclose(lagrange.coeffs, newton.coeffs)

Вариант a
Xi: [-3. -1.  1.  3.]
Yi: [2.8198421  2.35619449 0.78539816 0.32175055]
Разделенные разности: [ 2.8198421  -0.2318238  -0.13839359  0.0461312 ]
Многочлен Лагранжа:
         3
0.04613 x - 0.8315 x + 1.571
Многочлен Ньютона, приведенный к стандартному виду:
         3
0.04613 x - 0.8315 x + 1.571
P(X*) = 1.9807946072
f(X*) = 2.0344439358
|f(X*) - P(X*)| = 0.0536493286

Вариант б
Xi: [-3.  0.  1.  3.]
Yi: [2.8198421  1.57079633 0.78539816 0.32175055]
Разделенные разности: [ 2.8198421  -0.41634859 -0.09226239  0.0461312 ]
Многочлен Лагранжа:
         3
0.04613 x - 0.8315 x + 1.571
Многочлен Ньютона, приведенный к стандартному виду:
         3
0.04613 x - 0.8315 x + 1.571
P(X*) = 1.9807946072
f(X*) = 2.0344439358
|f(X*) - P(X*)| = 0.0536493286



# 3.2. Кубический сплайн

Построить кубический сплайн по таблице, считая, что кривизна равна нулю при $x = x_0$ и $x = x_4$.

$X^* = -0.5$.

| i | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| $x_i$ | -3.0 | -1.0 | 1.0 | 3.0 | 5.0 |
| $f_i$ | 2.8198 | 2.3562 | 0.78540 | 0.32175 | 0.19740 |

In [3]:
spline_x = np.array([-3.0, -1.0, 1.0, 3.0, 5.0])
spline_y = np.array([2.8198, 2.3562, 0.78540, 0.32175, 0.19740])

coefficients = natural_cubic_spline_coefficients(spline_x, spline_y)
a, b, c, d = coefficients

print("Коэффициенты S_i(x) = a_i + b_i(x - x_i) + c_i(x - x_i)^2 + d_i(x - x_i)^3")
for i in range(len(a)):
    print(
        f"[{spline_x[i]}, {spline_x[i + 1]}]: "
        f"a={a[i]:.10f}, b={b[i]:.10f}, c={c[i]:.10f}, d={d[i]:.10f}"
    )

spline_value = evaluate_cubic_spline(spline_x, coefficients, x_star)
print(f"S({x_star}) = {spline_value:.10f}")

Коэффициенты S_i(x) = a_i + b_i(x - x_i) + c_i(x - x_i)^2 + d_i(x - x_i)^3
[-3.0, -1.0]: a=2.8198000000, b=-0.0470026786, c=0.0000000000, d=-0.0461993304
[-1.0, 1.0]: a=2.3562000000, b=-0.6013946429, c=-0.2771959821, d=0.0925966518
[1.0, 3.0]: a=0.7854000000, b=-0.5990187500, c=0.2783839286, d=-0.0473935268
[3.0, 5.0]: a=0.3217500000, b=-0.0542053571, c=-0.0059772321, d=0.0009962054
S(-0.5) = 1.9977782645


Итоговые контрольные значения:

- для 3.1: $P(-0.5) \approx 1.9807946072$, $f(-0.5) \approx 2.0344439358$, погрешность $\approx 0.0536493286$;
- для 3.2: $S(-0.5) \approx 1.9977782645$.